In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
import os
import sys

%matplotlib inline

from _helpers import mock_snakemake
snakemake = mock_snakemake(
    "plot_mga",
)

# MGA Analysis: HBI Production by Political Stability

### Load snakemake and configuration

In [ ]:
import os
import glob
import pandas as pd
import pypsa
from pypsa import Network

# Access MGA scenario configuration from snakemake.config
mga_config = snakemake.config.get("scenario", {}).get("mga-new-indicators-multiple", {})
mga_modifiers = mga_config.get("modifiers", {})
mga_slack_values = mga_modifiers.get("mga", {}).get("slack", [])
cost_year = "2050"
interone = "hbi"
intertwo = "eaf-grid"
final = "steel"
scenario = "mga-new-indicators-multiple"

print(f"MGA Configuration: {mga_config}")
print(f"MGA Modifiers: {mga_modifiers}")
print(f"Slack values from config: {mga_slack_values}")

### Load PyPSA networks using NetworkCollection

In [ ]:
# Construct path to results directory
results_dir = os.path.abspath(os.path.join(
    os.path.dirname(snakemake.input.network_mga_production),
    "..",
    scenario
))

if not os.path.exists(results_dir):
    results_dir = os.path.abspath(os.path.join(
        os.path.dirname(snakemake.input.network_mga_production)
    ))

print(f"Results directory: {results_dir}")

# Load networks into a NetworkCollection
networks_dict = {}
slack_values = [0] + mga_slack_values

print(f"\nLoading networks for slack values: {slack_values}")

for slack in slack_values:
    if slack == 0:
        filepath = os.path.join(results_dir, "network_nan.nc")
        slack_key = None
    else:
        filepath = os.path.join(results_dir, f"network_{slack}.nc")
        slack_key = slack
    
    if os.path.exists(filepath):
        try:
            networks_dict[slack_key] = pypsa.Network(filepath)
            label = "optimal solution (slack=0)" if slack == 0 else f"slack={slack}"
            print(f"  ✓ Loaded {label}")
        except Exception as e:
            print(f"  ✗ Error loading {filepath}: {e}")
    else:
        print(f"  ✗ File not found: {filepath}")

# Create NetworkCollection from the loaded networks
# (Store as dict for now, can use NetworkCollection if available in this PyPSA version)
networks = networks_dict

print(f"\n✓ Successfully loaded {len(networks_dict)} networks")
print(f"✓ Network keys: {sorted(networks_dict.keys(), key=lambda x: (x is None, x))}")

### Prepare MGA criteria: Political Stability

In [ ]:
stability_file = snakemake.input.political_stability

print(f"Loading political stability from: {stability_file}")
stability_df = pd.read_csv(stability_file, index_col=0)
print("Political stability data:")
print(stability_df)

# Define threshold from MGA config
threshold_stability = 30

# Separate stable and unstable regions
stable_regions = stability_df[stability_df.iloc[:, 0] >= threshold_stability].index.tolist()
unstable_regions = stability_df[stability_df.iloc[:, 0] < threshold_stability].index.tolist()

print(f"\nStable regions (stability >= {threshold_stability}): {stable_regions}")
print(f"Unstable regions (stability < {threshold_stability}): {unstable_regions}")

all_regions = stable_regions + unstable_regions
print(f"\nTotal regions: {len(all_regions)}")
for region in sorted(all_regions):
    is_stable = region in stable_regions
    print(f"  {region}: {'STABLE' if is_stable else 'UNSTABLE'}")

### Read PyPSA networks and extract HBI production data

In [ ]:
# Sort network keys: None (optimal) first, then MGA variants in ascending order
sorted_keys = sorted(networks_dict.keys(), key=lambda x: (x is not None, x))

# Create epsilon array for columns (0 for optimal, then slack values)
epsilon_values = [0 if k is None else k for k in sorted_keys]

# Initialize storage for regional HBI production
hbi_production_data = {region: [] for region in all_regions}

# Extract HBI production from each network
for net_key in sorted_keys:
    n = networks_dict[net_key]
    
    # Initialize regional totals for this network
    regional_totals = {region: 0 for region in all_regions}
    
    # Get HBI production from links (input side: bus0, p0)
    # Note: HBI links have ore as bus0 and HBI as bus1, with efficiency factor
    # The actual production flows through p0 (input side of the conversion)
    links_df = n.links
    
    for link_idx, link_row in links_df.iterrows():
        if link_row['carrier'] == 'hbi':
            bus1 = link_row['bus1']  # Output bus (e.g., "Europe_hbi")
            
            # Get the HBI production from this link
            # Use p0 (input ore amount) to track production
            if hasattr(n, 'links_t') and 'p0' in n.links_t:
                p_sum = n.links_t['p0'][link_idx].sum()
            else:
                p_sum = link_row.get('p_nom_opt', 0)
            
            if not pd.isna(p_sum) and p_sum > 0:
                # Determine which region this HBI output belongs to
                # Extract region from bus1 (e.g., "Europe_hbi" -> "Europe")
                found = False
                for region in all_regions:
                    if bus1.startswith(region):
                        regional_totals[region] += p_sum
                        found = True
                        break
            
            
    
    # Append regional totals for this network
    for region in all_regions:
        hbi_production_data[region].append(regional_totals[region])



# Create pandas DataFrame with regions as rows and slack values as columns
# Convert from tonnes to Mt (divide by 1 million)
df_hbi = pd.DataFrame(hbi_production_data, index=epsilon_values).T / 1e6

# Adjust from input (iron ore) to output (HBI)
df_hbi = df_hbi / snakemake.config["iron_ore"]["ore_to_steel_ratio"]

# Rename index to be more descriptive
df_hbi.index.name = 'Region'
df_hbi.columns.name = 'MGA Slack (ε)'

print("HBI Production by Region and MGA Slack (in Mt):")
df_hbi.round(1)

### Append MGA criteria (political stability) to the dataframe

In [ ]:
# Add political stability column
df_hbi['Political Stability'] = stability_df.iloc[:, 0]

# Add stability classification (Stable/Unstable)
df_hbi['Stability Class'] = df_hbi.index.map(
    lambda region: 'Stable' if region in stable_regions else 'Unstable'
)
df_hbi

### Plot HBI production data

In [ ]:

# Extract epsilon values and production data
epsilon_cols = df_hbi.columns[:-2]  # All columns except 'Political Stability' and 'Stability Class'
epsilon_vals = np.array([float(x) for x in epsilon_cols])

# Aggregate by stability class
total_stable_mt = df_hbi[df_hbi['Stability Class'] == 'Stable'][epsilon_cols].sum()
total_unstable_mt = df_hbi[df_hbi['Stability Class'] == 'Unstable'][epsilon_cols].sum()

# Convert to numpy arrays
total_stable_mt = np.array(total_stable_mt.values, dtype=float)
total_unstable_mt = np.array(total_unstable_mt.values, dtype=float)

# Get individual regional data
stable_production_mt = {region: np.array(df_hbi.loc[region, epsilon_cols].values, dtype=float)
                        for region in stable_regions if region in df_hbi.index}
unstable_production_mt = {region: np.array(df_hbi.loc[region, epsilon_cols].values, dtype=float)
                          for region in unstable_regions if region in df_hbi.index}

# Load region colors from snakemake config
region_colors = snakemake.config["colors"]

# Create single figure
fig, ax = plt.subplots(1, 1, figsize=(12, 7))

# Settings
GAP_FRAC = 0.04

# ── Plot global (aggregated) line for stable regions (thicker, more prominent) ────
stable_color = "#3B6D11"
ax.plot(epsilon_vals, total_stable_mt, color=stable_color, linewidth=3, marker='o', 
        markersize=8, label="Global (Stable)", zorder=20)

# ── Plot all individual regional lines ────────────────────────────────────────

# Stable regions (solid lines)
for region in stable_regions:
    if region in stable_production_mt:
        region_data = stable_production_mt[region]
        if region_data.sum() > 0:  # Only plot if region has any production
            color = region_colors.get(region, '#000000')
            ax.plot(epsilon_vals, region_data, color=color, linewidth=1.2, 
                   linestyle='-', alpha=0.6, label=region, zorder=5)

# Unstable regions (dashed lines)
for region in unstable_regions:
    if region in unstable_production_mt:
        region_data = unstable_production_mt[region]
        if region_data.sum() > 0:  # Only plot if region has any production
            color = region_colors.get(region, '#000000')
            ax.plot(epsilon_vals, region_data, color=color, linewidth=1.2, 
                   linestyle='--', alpha=0.7, label=region, zorder=5)

# ── Axes formatting ──────────────────────────────────────────────────────────
ax.set_xlim(min(epsilon_vals) - 0.002, max(epsilon_vals) + 0.01)
ax.set_ylim(0, max(total_stable_mt) * 1.15)
ax.set_xlabel("MGA slack (fraction, ε)", fontsize=12)
ax.set_ylabel("HBI production in Mt", fontsize=12)
ax.set_title("HBI Production by Exporter Stability (All 16 Regions)")
ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)
ax.legend(loc="best", fontsize=7, framealpha=0.95, ncol=3, title="Regions")

plt.tight_layout()
plt.savefig(snakemake.output.mga_plot, dpi=300, bbox_inches="tight")
plt.savefig(snakemake.output.mga_plot_png, dpi=300, bbox_inches="tight")
plt.show()

### Statistics

In [ ]:

print("✓ Plot with all 16 regions created successfully!")
print(f"\nStable regions HBI production range: {total_stable_mt.min():.4f} to {total_stable_mt.max():.4f} Mt")
if total_stable_mt[0] > 0:
    print(f"  Change: {((total_stable_mt[-1] / total_stable_mt[0]) - 1) * 100:.1f}%")
print(f"\nUnstable regions HBI production range: {total_unstable_mt.min():.4f} to {total_unstable_mt.max():.4f} Mt")
if total_unstable_mt[0] > 0:
    print(f"  Change: {((total_unstable_mt[-1] / total_unstable_mt[0]) - 1) * 100:.1f}%")

---
# MGA Analysis: HBI Trade through Maritime Chokepoints

Analyse how much HBI shipping trade passes through each maritime chokepoint using the **mga-chokepoints** scenario networks. This scenario minimises trade through chokepoint-exposed routes.

### Load chokepoint data and build route-to-chokepoint mapping

In [ ]:
# ── Load MGA-chokepoints scenario configuration ──
cp_scenario = "mga-chokepoints"
cp_mga_config = snakemake.config.get("scenario", {}).get(cp_scenario, {})
cp_mga_modifiers = cp_mga_config.get("modifiers", {})
cp_mga_slack_values = cp_mga_modifiers.get("mga", {}).get("slack", [])

# Chokepoints listed in the config
config_chokepoints = cp_mga_modifiers.get("mga", {}).get("threshold_value", [])

print(f"Chokepoint MGA scenario: {cp_scenario}")
print(f"Slack values: {cp_mga_slack_values}")
print(f"Chokepoints from config: {config_chokepoints}")


# ── Load PyPSA networks from mga-chokepoints results ──
cp_results_dir = os.path.abspath(os.path.join(
        os.path.dirname(snakemake.input.network_mga_chokepoints)
    ))

print(f"\nResults directory: {cp_results_dir}")

cp_networks_dict = {}
cp_slack_values = [0] + cp_mga_slack_values

for slack in cp_slack_values:
    if slack == 0:
        filepath = os.path.join(cp_results_dir, "network_nan.nc")
        slack_key = None
    else:
        filepath = os.path.join(cp_results_dir, f"network_{slack}.nc")
        slack_key = slack

    if os.path.exists(filepath):
        try:
            cp_networks_dict[slack_key] = pypsa.Network(filepath)
            label = "optimal (slack=0)" if slack == 0 else f"slack={slack}"
            print(f"  ✓ Loaded {label}")
        except Exception as e:
            print(f"  ✗ Error loading {filepath}: {e}")
    else:
        print(f"  ✗ File not found: {filepath}")

print(f"\n✓ Loaded {len(cp_networks_dict)} chokepoint MGA networks")

# ── Sort keys and build epsilon array ──
cp_sorted_keys = sorted(cp_networks_dict.keys(), key=lambda x: (x is not None, x))
cp_epsilon_values = [0 if k is None else k for k in cp_sorted_keys]

# ── Load trade options with chokepoints column ──
trade_options_cp = pd.read_csv(snakemake.input.trade_options_chokepoints, header=0)

# Build mapping: (region_from, region_to) -> set of chokepoints
route_chokepoints = {}
for _, row in trade_options_cp.iterrows():
    r_from = row["region_from"]
    r_to = row["region_to"]
    cp_str = row.get("chokepoints", "")
    if pd.notna(cp_str) and cp_str != "":
        cp_set = set(str(cp_str).split(";"))
    else:
        cp_set = set()
    route_chokepoints[(r_from, r_to)] = cp_set

# Show summary: how many routes pass through each chokepoint
from collections import Counter
cp_count = Counter()
for cp_set in route_chokepoints.values():
    for cp in cp_set:
        if cp in config_chokepoints:
            cp_count[cp] += 1

print(f"\nRoutes per chokepoint (config only):")
for cp in config_chokepoints:
    print(f"  {cp}: {cp_count.get(cp, 0)} routes")

### Extract HBI shipping trade through chokepoints per MGA slack

In [ ]:
# For each chokepoint MGA network, sum HBI shipping trade through each chokepoint
# A link contributes to a chokepoint if its route passes through it.
# p_nom_opt gives the capacity allocated to each shipping link.

carrier = "hbi"

# Initialize: {chokepoint: [trade_per_epsilon...]}
cp_trade_data = {cp: [] for cp in config_chokepoints}
cp_trade_total = []  # total trade through any config chokepoint

for net_key in cp_sorted_keys:
    n_cp = cp_networks_dict[net_key]

    # Per-chokepoint totals for this network
    cp_totals = {cp: 0.0 for cp in config_chokepoints}
    total_through_any = 0.0

    # Iterate shipping HBI links
    shipping_links = n_cp.links[n_cp.links.carrier == f"shipping_{carrier}"]

    for link_name, link_row in shipping_links.iterrows():
        # Extract region_from, region_to from link name: "shipping hbi {r_from}-{r_to}"
        route_part = link_name.replace(f"shipping {carrier} ", "")
        parts = route_part.split("-")
        if len(parts) != 2:
            continue
        r_from, r_to = parts

        # Get trade volume (p_nom_opt = capacity in t)
        trade_vol = link_row.get("p_nom_opt", 0)
        if pd.isna(trade_vol) or trade_vol <= 0:
            continue

        # Check which config chokepoints this route passes through
        route_cp = route_chokepoints.get((r_from, r_to), set())
        matched_cp = route_cp & set(config_chokepoints)

        if matched_cp:
            total_through_any += trade_vol
            for cp in matched_cp:
                cp_totals[cp] += trade_vol

    cp_trade_total.append(total_through_any)
    for cp in config_chokepoints:
        cp_trade_data[cp].append(cp_totals[cp])

# Build DataFrame: rows = chokepoints, columns = epsilon values
df_cp = pd.DataFrame(cp_trade_data, index=cp_epsilon_values).T / 1e6  # Convert to Mt
df_cp.index.name = "Chokepoint"
df_cp.columns.name = "MGA Slack (ε)"

# Add total row
total_series = pd.Series(cp_trade_total, index=cp_epsilon_values, name="TOTAL (any chokepoint)") / 1e6

print("HBI shipping trade through chokepoints (Mt):")
df_cp_display = pd.concat([df_cp, total_series.to_frame().T])
df_cp_display.round(1)

### Plot: HBI trade through chokepoints vs MGA slack

In [ ]:
epsilon_arr = np.array(cp_epsilon_values, dtype=float)

# Filter to chokepoints that actually have non-zero trade
active_cp = [cp for cp in config_chokepoints if df_cp.loc[cp].sum() > 0]
print(f"Active chokepoints (with trade): {active_cp}")

# Chokepoint colors (distinct palette)
cp_colors = {
    "babalmandab": "#e41a1c",
    "bosporus":    "#377eb8",
    "gibraltar":   "#4daf4a",
    "suez":        "#984ea3",
    "panama":      "#ff7f00",
    "ormuz":       "#a65628",
    "northwest":   "#f781bf",
    "malacca":     "#999999",
    "sunda":       "#66c2a5",
    "chili":       "#8dd3c7",
    "south_africa":"#fdb462",
}

fig, ax = plt.subplots(1, 1, figsize=(12, 7))

# ── Plot total (aggregated) as thick black line ──
total_arr = np.array(cp_trade_total, dtype=float) / 1e6
ax.plot(epsilon_arr, total_arr, color="black", linewidth=3, marker="o",
        markersize=8, label="Total (any chokepoint)", zorder=20)

# ── Plot individual chokepoints (disaggregated) ──
for cp in active_cp:
    cp_arr = np.array(df_cp.loc[cp].values, dtype=float)
    color = cp_colors.get(cp, "#333333")
    label = cp.replace("_", " ").title()
    ax.plot(epsilon_arr, cp_arr, color=color, linewidth=1.5,
            linestyle="--", marker="s", markersize=4, alpha=0.8,
            label=label, zorder=10)

# ── Axes formatting ──
ax.set_xlim(min(epsilon_arr) - 0.002, max(epsilon_arr) + 0.01)
ax.set_ylim(0, max(total_arr) * 1.15)
ax.set_xlabel("MGA slack (fraction, ε)", fontsize=12)
ax.set_ylabel("HBI shipping trade (Mt)", fontsize=12)
ax.set_title("HBI Shipping Trade Through Maritime Chokepoints vs MGA Slack (mga-chokepoints)", fontsize=13)
ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)
ax.legend(loc="best", fontsize=8, framealpha=0.95, ncol=2, title="Chokepoint")

plt.tight_layout()
plt.show()

# Print summary
if total_arr[0] > 0:
    print(f"\nTotal chokepoint-exposed trade: {total_arr[0]:.1f} Mt (optimal) → {total_arr[-1]:.1f} Mt (ε={epsilon_arr[-1]})")
    print(f"  Change: {((total_arr[-1] / total_arr[0]) - 1) * 100:.1f}%")

# Misc

In [ ]:
n.statistics.energy_balance().div(1e6)

In [ ]:
n = cp_networks_dict[None] # None for optimal solution, or use specific slack value if desired

In [ ]:
# Total shipping volume
n.links[n.links.carrier == "shipping_hbi"].p_nom_opt.div(1e6).sum().round(1)

In [ ]:
# Top shipping routes
n.links[n.links.carrier == "shipping_hbi"].p_nom_opt.div(1e6).sort_values(ascending=False).round(1).head(20)

In [ ]:
# Top shipping routes: costs
n.links[n.links.carrier == "shipping_hbi"].marginal_cost

#p_nom_opt.div(1e6).sort_values(ascending=False).round(1).head(20)

In [ ]:
# Top shipping routes: costs
n.links[n.links.carrier == "shipping_hbi"].efficiency

#p_nom_opt.div(1e6).sort_values(ascending=False).round(1).head(20)